## How to run this notebook

There are two ways. Both are safe; choose either.

**Way 1: run your own copy.** No warning, and your changes are kept.

- At the top of this page, choose **Copy to Drive** (or **File → Save a copy in Drive**).
- Your copy opens in a new tab. It is saved in your Google Drive, in the folder **Colab Notebooks**.
- In your copy, choose **Runtime → Run all**.

**Way 2: run this page directly.** Quicker, but your changes are not kept.

- Choose **Runtime → Run all**.
- Colab shows this warning:

  > Warning: This notebook was not authored by Google. This notebook is being loaded from GitHub. It may request access to your data stored with Google, or read data and credentials from other sessions. Please review the source code before executing this notebook.

- Choose **Run anyway**.

**Why it is safe.** Colab shows this warning for every notebook that does not come from Google, before it has looked at the code. This notebook does not ask for your Google account, your Drive or any other data of yours. It only saves the table it analyses, and any figures it draws, in this session, which ends when you close the page. You can read every line of the code below before you run it.

# Delivery times: Sturges' classes and a normal fit test

**Frequency Distribution and Histogram** · a worked example from the StatNex teaching guide

**The story.** The table holds the delivery times in minutes of 50 orders from each of three food-delivery apps: ZipEats, FeastNow and UrbanBite. Marketing wants a simple picture of "how long you usually wait" with each app.

**The question.** How are delivery times distributed for each app, and where do most orders fall?

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## Before the analysis

In [ ]:
# Each script saves its figures to the folder figures/, as it would on your computer.
# This shows each new figure under the cell that saved it.
from pathlib import Path

import IPython.display
import matplotlib

matplotlib.use("Agg")


def show_new_figures(*_, shown=set()):
    for path in sorted(Path("figures").glob("*.png"), key=lambda p: p.stat().st_mtime_ns):
        if (path.name, path.stat().st_mtime_ns) not in shown:
            shown.add((path.name, path.stat().st_mtime_ns))
            IPython.display.display(IPython.display.Image(filename=str(path)))


events = IPython.get_ipython().events
for old in [f for f in events.callbacks["post_run_cell"] if f.__name__ == "show_new_figures"]:
    events.unregister("post_run_cell", old)
events.register("post_run_cell", show_new_figures)

## The table

This cell saves the table `01_basic/frequency_histogram.csv` (50 rows) as `data.csv`, the file the code reads.

In [ ]:
with open("data.csv", "w", encoding="utf-8") as file:
    file.write("""\
No.,ZipEats,FeastNow,UrbanBite
1.0,16.0,32.0,36.0
2.0,24.0,26.0,41.0
3.0,28.0,31.0,38.0
4.0,19.0,33.0,48.0
5.0,29.0,25.0,33.0
6.0,26.0,37.0,42.0
7.0,22.0,30.0,42.0
8.0,30.0,28.0,42.0
9.0,22.0,29.0,34.0
10.0,26.0,35.0,46.0
11.0,28.0,33.0,38.0
12.0,17.0,27.0,53.0
13.0,29.0,34.0,41.0
14.0,26.0,26.0,45.0
15.0,23.0,23.0,35.0
16.0,23.0,30.0,46.0
17.0,23.0,30.0,47.0
18.0,22.0,33.0,33.0
19.0,26.0,35.0,44.0
20.0,12.0,29.0,31.0
21.0,26.0,29.0,32.0
22.0,14.0,42.0,44.0
23.0,29.0,33.0,42.0
24.0,20.0,32.0,39.0
25.0,23.0,32.0,41.0
26.0,20.0,41.0,38.0
27.0,27.0,29.0,46.0
28.0,23.0,36.0,38.0
29.0,28.0,34.0,36.0
30.0,19.0,33.0,31.0
31.0,27.0,30.0,36.0
32.0,27.0,32.0,38.0
33.0,29.0,38.0,32.0
34.0,24.0,35.0,46.0
35.0,26.0,41.0,45.0
36.0,18.0,31.0,38.0
37.0,25.0,37.0,44.0
38.0,58.0,37.0,34.0
39.0,17.0,33.0,32.0
40.0,31.0,31.0,29.0
41.0,27.0,40.0,45.0
42.0,18.0,39.0,40.0
43.0,24.0,35.0,37.0
44.0,29.0,37.0,42.0
45.0,25.0,35.0,43.0
46.0,24.0,35.0,36.0
47.0,25.0,31.0,39.0
48.0,23.0,33.0,40.0
49.0,20.0,37.0,46.0
50.0,26.0,33.0,44.0
""")

## Libraries

In [ ]:
import re
from decimal import Decimal
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

pd.set_option("display.width", 120)

## The data

In [ ]:
# One column per variable. Blank and text cells are left out, so columns may differ
# in length.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
columns = ["ZipEats", "FeastNow", "UrbanBite"]
series = {c: pd.to_numeric(data[c], errors="coerce").dropna().to_numpy(float) for c in columns}

## Descriptive statistics

In [ ]:
# Variance and SD divide by n - 1; quartiles interpolate linearly between the sorted
# values (NumPy's default, Excel's QUARTILE.INC). The fences Q1 - 1.5 IQR and
# Q3 + 1.5 IQR mark where a value starts to count as an outlier.
def describe(x):
    q1, median, q3 = np.percentile(x, [25, 50, 75])
    mean, sd = np.mean(x), np.std(x, ddof=1)
    return {"n": x.size, "mean": mean, "variance": np.var(x, ddof=1), "sd": sd,
            "min": x.min(), "max": x.max(), "cv": sd / mean if mean else np.nan,
            "Q1 - 1.5 IQR": q1 - 1.5 * (q3 - q1), "Q1": q1, "median": median, "Q3": q3,
            "Q3 + 1.5 IQR": q3 + 1.5 * (q3 - q1)}


summary = pd.DataFrame({name: describe(x) for name, x in series.items()})
print(summary)

## Classes

In [ ]:
# Boundary i is start + i × width, rounded to the decimals the start and the width are
# written with: floating point makes 0.1 × 3 0.30000000000000004, which would put a
# value of 0.3 below its own boundary and count it a class low.
def decimals(x):
    """How many decimals x is written with, at 15 significant digits: 0.1 has 1, 20 has 0."""
    return max(0, -Decimal(f"{x:.15g}").as_tuple().exponent)


def boundary(start, width, i):
    return round(float(start + i * width), max(decimals(start), decimals(width)))


# The range from the smallest to the largest value is worked out to the decimals they
# are written with: floating point makes 0.8 - 0.6 0.20000000000000007, not 0.2.
def data_range(x):
    return round(float(x.max() - x.min()), max(decimals(x.min()), decimals(x.max())))


# Sturges' formula fixes the number of classes, k = ceil(log2(n)) + 1, and the range
# from the smallest to the largest value is shared equally between them. The width is
# not rounded: this is the formula itself (NumPy's bins="sturges" is the same rule).
# It assumes roughly bell-shaped data and gives too few classes for large samples.
def class_count(x):
    return int(np.ceil(np.log2(x.size))) + 1


def class_edges(x):
    count = class_count(x)
    span = data_range(x)
    width = span / count if span > 0 else 1.0
    edges = np.array([boundary(x.min(), width, i) for i in range(count + 1)])
    # Rounding can leave the last boundary a hair below the largest value.
    edges[-1] = max(edges[-1], x.max())
    return edges

## Frequency tables

In [ ]:
# Classes are [lower, upper): a value on a boundary belongs to the class above, except
# that the last class also holds its upper bound.
frequencies, edges_of = {}, {}
for name, x in series.items():
    edges = class_edges(x)
    edges_of[name] = edges
    counts = np.histogram(x, bins=edges)[0]
    relative = counts / x.size
    frequencies[name] = pd.DataFrame({"lower": edges[:-1], "frequency": counts,
                                      "relative": relative, "cumulative": np.cumsum(relative)})
    print(f"\n{name}: class boundaries {np.round(edges, 6).tolist()}")
    print(frequencies[name].to_string(index=False))

## Goodness of fit

In [ ]:
# Expected counts under a normal distribution with the data's mean and SD (n - 1).
# The two end classes are stretched to minus and plus infinity, so the probabilities
# add up to 1. Two parameters were estimated, so df = classes - 3 (the textbook rule;
# strictly, with the mean and SD taken from the raw values rather than the counts,
# the true reference lies between chi-square on classes - 3 and classes - 1 df).
def probabilities(x, edges):
    cut = np.array(edges, dtype=float)
    cut[0], cut[-1] = -np.inf, np.inf
    return np.diff(stats.norm.cdf(cut, loc=x.mean(), scale=x.std(ddof=1))), 2


# Pearson's chi-square, sum of (observed - expected)² / expected: stats.chisquare,
# whose ddof takes off one degree of freedom per estimated parameter. A large value
# (small P) says the counts do not follow the curve. Read the table first: one class
# with a tiny expected count can supply almost all of the chi-square. The usual rule
# of thumb asks for an expected count of at least 5 in every class; below that the P
# value is rough, so merge sparse end classes. And a large P is not proof that the
# curve fits, only that these counts cannot rule it out. With df of 0 or less there
# is no test, and the P value is left empty.
expected_tables, fits = {}, {}
for name, x in series.items():
    observed = frequencies[name]["frequency"].to_numpy()
    p, estimated = probabilities(x, edges_of[name])
    expected = p * x.size
    df = observed.size - 1 - estimated
    test = stats.chisquare(observed, expected, ddof=estimated)
    expected_tables[name] = pd.DataFrame({"lower": edges_of[name][:-1], "observed": observed,
                                          "expected": expected})
    fits[name] = pd.Series({"chi-square": test.statistic, "df": df,
                            "P": test.pvalue if df > 0 else np.nan})
    print(f"\n{name}")
    print(expected_tables[name].to_string(index=False))
    print(fits[name].to_string())

## Histograms

In [ ]:
Path("figures").mkdir(exist_ok=True)
fig, axes = plt.subplots(len(series), 1, figsize=(7, 3 * len(series)), squeeze=False)
for ax, (name, table) in zip(axes[:, 0], frequencies.items()):
    widths = np.diff(edges_of[name])
    # With unequal classes the area, not the height, carries the count: frequency density.
    unequal = not np.allclose(widths, widths[0])
    ax.bar(table["lower"], table["frequency"] / widths if unequal else table["frequency"],
           width=widths, align="edge", edgecolor="black")
    ax.set_title(name)
    ax.set_ylabel("Frequency per unit width" if unequal else "Frequency")
fig.tight_layout()
fig.savefig("figures/histogram.png", dpi=150)
plt.close(fig)

## What the result says

Sturges' formula chooses the number of classes from the sample size. The chi-square goodness-of-fit test then compares the observed count in each class with the count a normal distribution (a bell curve) would give.

For FeastNow, chi-square = 1.34 and P = 0.854. For UrbanBite, chi-square = 5.57 and P = 0.234. The P value is the chance of a gap this large if the data really were normal. Both are large, so a bell curve is not ruled out.

For ZipEats, chi-square = 1236 and P is practically zero. One order sits in the last class, where a bell curve expects almost none (0.0008). That lone 58-minute order is why ZipEats fails.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.